In [1]:
"""
Estensione di percp_attack_battery.py:
  1) supporto multi-dataset: CIFAR-10, CIFAR-100, ImageNet
  2) batteria di attacchi aggiornata per il solo threat model L_infty:
     - RIMOSSI: EAD (nativo L1), DDN (nativo L2), ALMA (nativo L2/L1)
       -> non hanno senso in un confronto Linf-only, meglio toglierli
          che proiettarli forzatamente
     - AGGIUNTI: FGSM, BIM (nativi Linf), DeepFool (nativo L2, va
       proiettato sulla palla Linf come gia' si faceva per CW/FAB)

Questo file e' AUTOCONTENUTO: sostituisce (non aggiunge a) l'ATTACK_FUNCS
di percp_attack_battery.py. QuantileRegressor/ProbabilityRegressionDataset/
cqr_loss/evaluate_attack_percp_v2 restano quelli che hai gia' nel notebook:
funzionano invariati su qualunque dataset perche' QuantileRegressor usa un
resnet18 con global average pooling (nessun flatten a dimensione fissa),
quindi non serve toccarlo per passare da 32x32 (CIFAR) a 224x224 (ImageNet).
"""

import os
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
import torchattacks
from tqdm import tqdm
from secml.utils import fm
from secml import settings
from robustbench.utils import load_model as _rb_load_model

import torch.nn.functional as F
from secml.utils import fm
import numpy as np

import matplotlib.pyplot as plt

from nonconformist.cp import IcpRegressor
from nonconformist.nc import RegressorNc, QuantileRegErrFunc

import foolbox as fb
import pandas as pd


# ----------------------------------------------------------------------
# 1) DATASET
# ----------------------------------------------------------------------
# eps di default = quelli usati nel leaderboard RobustBench per ciascun
# dataset in norma Linf. Per ImageNet la risoluzione e' 224x224 e il budget
# e' piu' piccolo (4/255 contro 8/255 di CIFAR) perche' l'immagine ha molti
# piu' pixel: un eps=8/255 su 224x224 sarebbe un attacco molto piu' forte
# che sulla 32x32 di CIFAR.
DATASET_DEFAULT_EPS_LINF = {
    "CIFAR10": 8 / 255,
    "CIFAR100": 8 / 255,
    "IMAGENET": 4 / 255,
}


def load_dataset(name, root, train=True, imagenet_split=""):
    """Ritorna un torch Dataset che restituisce (img_tensor in [0,1], label).
    Nessuna Normalize: i modelli RobustBench normalizzano internamente,
    e gli attacchi (torchattacks/foolbox) si aspettano input in [0,1].
    """
    name = name.upper()

    if name == "CIFAR10":
        transform = T.ToTensor()
        return torchvision.datasets.CIFAR10(
            root=root, train=train, download=True, transform=transform
        )

    if name == "CIFAR100":
        transform = T.ToTensor()
        return torchvision.datasets.CIFAR100(
            root=root, train=train, download=True, transform=transform
        )

    if name == "IMAGENET":
        # ImageNet va scaricato a mano (licenza); qui assumo tu abbia gia'
        # la cartella con la struttura ImageFolder standard, cioe':
        #   root/val/n01440764/xxx.JPEG, root/val/n01443537/yyy.JPEG, ...
        # Se invece hai il devkit ufficiale e vuoi usare
        # torchvision.datasets.ImageNet(root, split='val'), sostituisci
        # la riga sotto - il resto del codice non cambia, l'importante e'
        # che __getitem__ ritorni (img_tensor_[0,1], label_int).
        transform = T.Compose([
            T.Resize(256),
            T.CenterCrop(224),
            T.ToTensor(),
        ])
        split_dir = os.path.join(root, imagenet_split)
        return torchvision.datasets.ImageFolder(split_dir, transform=transform)

    raise ValueError(f"Dataset {name} non supportato. Usa CIFAR10, CIFAR100 o IMAGENET.")


# ----------------------------------------------------------------------
# 2) MODELLO ROBUSTBENCH
# ----------------------------------------------------------------------
def load_robust_model(model_name, dataset, threat_model="Linf", device="cuda"):
    """Wrapper su robustbench.utils.load_model con l'API 'dataset' + 'threat_model'
    (quella corrente, non il vecchio 'norm=' usato nel notebook originale che
    e' rimasto legato implicitamente a CIFAR-10).

    Alcuni model_name Linf verificati sul model zoo di RobustBench, uno per
    dataset (puoi sostituirli con qualunque altro dalla leaderboard
    https://robustbench.github.io/ per quel dataset+threat_model):
      - cifar10:  'Wang2023Better_WRN-70-16'   (quello che stavi gia' usando)
      - cifar100: scegli un model_name dalla leaderboard CIFAR-100 (Linf) -
                  non ne inserisco uno di default per non rischiare di
                  indicarti un ID sbagliato/deprecato: controlla
                  https://robustbench.github.io/#div_cifar100_Linf_heading
      - imagenet: 'Salman2020Do_R18' (verificato, eps=4/255)
    """
    output_dir = fm.join(settings.SECML_MODELS_DIR, "robustbench")

    model = _rb_load_model(
        model_name=model_name,
        dataset=dataset.lower(),
        threat_model=threat_model,
        model_dir=output_dir,
    )
    model.eval()
    model.to(device)
    return model


# ----------------------------------------------------------------------
# 3) BATTERIA DI ATTACCHI - solo Linf
# ----------------------------------------------------------------------
def project_linf_ball(x_adv, x_orig, eps):
    delta = (x_adv - x_orig).clamp(-eps, eps)
    return (x_orig + delta).clamp(0, 1)


def _batched(atk, x, y, device, bs, desc=None):
    out = []
    for i in tqdm(range(0, len(x), bs), desc=desc or atk.__class__.__name__, leave=True):
        xb, yb = x[i:i + bs].to(device), y[i:i + bs].to(device)
        out.append(atk(xb, yb).detach().cpu())
    return torch.cat(out)


# --- fixed-budget (rispettano eps ad ogni step) -------------------------
def _run_fgsm(model, x, y, eps, steps, device, bs):
    atk = torchattacks.FGSM(model, eps=eps)
    return _batched(atk, x, y, device, bs, desc="FGSM")


def _run_bim(model, x, y, eps, steps, device, bs):
    alpha = eps / max(steps, 1)
    atk = torchattacks.BIM(model, eps=eps, alpha=alpha, steps=steps)
    return _batched(atk, x, y, device, bs, desc="BIM")


def _run_pgd(model, x, y, eps, steps, device, bs):
    alpha = (eps / steps) * 2
    atk = torchattacks.PGD(model, eps=eps, alpha=alpha, steps=steps, random_start=True)
    return _batched(atk, x, y, device, bs, desc="PGD")


def _run_apgd(model, x, y, eps, steps, device, bs):
    atk = torchattacks.APGD(model, norm="Linf", eps=eps, steps=steps, loss="ce")
    return _batched(atk, x, y, device, bs, desc="APGD")


def _run_aa(model, x, y, eps, steps, device, bs):
    atk = torchattacks.AutoAttack(model, norm="Linf", eps=eps, version="standard")
    return _batched(atk, x, y, device, bs, desc="AutoAttack")


# --- minimum-norm (proiettati sulla palla Linf dopo l'attacco) ----------
def _run_fab(model, x, y, eps, steps, device, bs):
    atk = torchattacks.FAB(model, norm="Linf", eps=eps, steps=steps, n_restarts=1, n_classes=10)
    x_adv = _batched(atk, x, y, device, bs, desc="FAB")
    return project_linf_ball(x_adv, x, eps)


def _run_cw(model, x, y, eps, steps, device, bs):
    # CW nativo e' L2: lo lasciamo cercare la sua perturbazione minima e poi
    # lo proiettiamo sulla palla Linf, esattamente come nella batteria precedente.
    atk = torchattacks.CW(model, c=1, kappa=0, steps=steps, lr=0.01)
    x_adv = _batched(atk, x, y, device, bs, desc="CW")
    return project_linf_ball(x_adv, x, eps)


def _run_deepfool(model, x, y, eps, steps, device, bs):
    # DeepFool e' nativamente L2 e non ha un parametro eps: proiezione
    # sulla palla Linf dopo l'attacco, stessa logica di CW/FAB.
    atk = torchattacks.DeepFool(model, steps=steps, overshoot=0.02)
    x_adv = _batched(atk, x, y, device, bs, desc="DeepFool")
    return project_linf_ball(x_adv, x, eps)


ATTACK_FUNCS = {
    "FGSM": _run_fgsm,
    "BIM": _run_bim,
    "PGD": _run_pgd,
    "APGD": _run_apgd,
    "AA": _run_aa,
    "FAB": _run_fab,
    "CW": _run_cw,
    "DEEPFOOL": _run_deepfool,
}


def generate_adversarial_examples(model, X, y, attack_name, eps, steps=100, bs=16, device="cuda"):
    """Dispatcher unico, solo Linf. Firma compatibile con evaluate_attack_percp_v2
    del file precedente (basta togliere l'argomento 'norm', non serve piu').
    """
    attack_name = attack_name.upper()
    if attack_name not in ATTACK_FUNCS:
        raise NotImplementedError(
            f"{attack_name} non riconosciuto. Disponibili: {list(ATTACK_FUNCS)} (+ 'WORST')"
        )
    return ATTACK_FUNCS[attack_name](model, X, y, eps, steps, device, bs)


def generate_worst_case_examples(model, X, y, get_true_probabilities,
                                  attack_names, eps, steps=100, bs=16, device="cuda"):
    """Come nella batteria precedente: per ogni punto tiene la versione con
    la p_true(x) piu' bassa tra gli attacchi in attack_names. Usa lo stesso
    pool/eps/steps in calibrazione e in test.
    """
    candidates, p_candidates = [], []
    for name in attack_names:
        print(f"\n--- Worst-case ensemble: genero {name} ---")
        X_adv = generate_adversarial_examples(
            model=model, X=X, y=y, attack_name=name, eps=eps, steps=steps, bs=bs, device=device
        )
        p_adv = get_true_probabilities(model, X_adv, y).view(-1).cpu()
        candidates.append(X_adv)
        p_candidates.append(p_adv)

    P = torch.stack(p_candidates, dim=0)
    worst_idx = P.argmin(dim=0)
    X_stack = torch.stack(candidates, dim=0)
    N = X.shape[0]
    X_worst = X_stack[worst_idx, torch.arange(N)]

    print("\nDistribuzione dell'attacco vincente per punto:")
    for i, name in enumerate(attack_names):
        pct = (worst_idx == i).float().mean().item() * 100
        print(f"  {name}: {pct:.1f}%")
    return X_worst


2026-07-22 10:00:23.424991: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-22 10:00:23.531090: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-22 10:00:25.662006: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
class RobustBenchProbabilityRegressor(nn.Module):

    def __init__(self, classifier):
        super().__init__()
        self.classifier = classifier


    def forward(self, x, y):

        logits = self.classifier(x)

        probs = F.softmax(logits, dim=1)

        idx = torch.arange(
            len(y),
            device=x.device
        )

        p_true = probs[idx,y]

        return p_true
    
def get_probability_targets(model,dataset,indices):

    X=[]
    y=[]
    p=[]


    with torch.no_grad():

        for i in indices:

            img,label=dataset[i]

            img=img.unsqueeze(0).cuda()
            label=torch.tensor(
                [label],
                device="cuda"
            )


            prob=model(
                img,
                label
            )


            X.append(img.cpu())
            y.append(label.cpu())
            p.append(prob.cpu())


    return (
        torch.cat(X),
        torch.cat(y),
        torch.cat(p)
    )

def get_true_probabilities(
    classifier,
    X,
    y,
    device="cuda"
):

    classifier.eval()

    probs_all=[]

    batch_size=64

    with torch.no_grad():

        for i in range(0,len(X),batch_size):

            x=X[i:i+batch_size].to(device)
            labels=y[i:i+batch_size].to(device)

            logits=classifier(x)

            probs=torch.softmax(
                logits,
                dim=1
            )

            p_true=probs[
                torch.arange(len(labels),device=device),
                labels
            ]

            probs_all.append(
                p_true.cpu()
            )

    return torch.cat(probs_all)

def split_train_calib(X_cal, y_cal, frac_train=0.5, seed=None):
    """Divide il pool 'X_cal' in due parti disgiunte:
    - X_train/y_train: usati per fittare il quantile regressor
    - X_calib/y_calib: usati SOLO per calcolare qhat (mai visti in training)
    """
    if seed is not None:
        torch.manual_seed(seed)
 
    n = len(X_cal)
    perm = torch.randperm(n)
    n_train = int(n * frac_train)
 
    idx_train = perm[:n_train]
    idx_calib = perm[n_train:]
 
    return (
        X_cal[idx_train], y_cal[idx_train],
        X_cal[idx_calib], y_cal[idx_calib],
    )
 

from torch.utils.data import Dataset, DataLoader


class ProbabilityRegressionDataset(Dataset):

    def __init__(self, X, y):
        self.X = X
        self.y = y.float()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return (
            self.X[idx],
            self.y[idx]
        )
    
import torchvision.models as models


class QuantileRegressor(nn.Module):

    def __init__(self):

        super().__init__()

        self.model = models.resnet18(
            weights=None
        )

        self.model.fc = nn.Linear(
            self.model.fc.in_features,
            2
        )


    def forward(self,x):

        out = self.model(x)

        return out
    
def pinball_loss(pred, target, quantile):

    error = target - pred

    return torch.mean(
        torch.maximum(
            quantile*error,
            (quantile-1)*error
        )
    )


def cqr_loss(pred, target, alpha):
    
    q_low=alpha/2
    q_high=1-alpha/2

    loss_low = pinball_loss(
        pred[:,0],
        target,
        q_low
    )

    loss_high = pinball_loss(
        pred[:,1],
        target,
        q_high
    )

    return loss_low + loss_high    

In [3]:
# ----------------------------------------------------------------------
# evaluate_attack_percp esteso: gestisce anche attack_name="WORST"
# (riusa lo split train/calib e il singolo qmodel del fix precedente)
# ----------------------------------------------------------------------
def evaluate_attack_percp_v2(
    attack_name,                # una tra ATTACK_FUNCS oppure "WORST"
    model_rb,
    QuantileRegressor,
    ProbabilityRegressionDataset,
    cqr_loss,
    get_true_probabilities,
    X_train, y_train,
    X_calib, y_calib,
    X_test, y_test,
    eps=8 / 255,
    alpha=0.1,
    steps=100,
    bs=16,
    epochs=50,
    device="cuda",
    worst_case_pool=("PGD", "APGD", "FAB", "CW"),  # sottoinsieme usato da "WORST"
):
    print(f"\nRunning {attack_name} (PERCP corretto)")

    # a) UN SOLO regressore, allenato SOLO su X_train
    p_train = get_true_probabilities(model_rb, X_train, y_train).view(-1)
    train_loader = torch.utils.data.DataLoader(
        ProbabilityRegressionDataset(X_train, p_train.cpu()), batch_size=32, shuffle=True
    )
    qmodel = QuantileRegressor().to(device)
    optimizer = torch.optim.Adam(qmodel.parameters(), lr=1e-4)
    for epoch in range(epochs):
        qmodel.train()
        total_loss = 0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            pred = qmodel(Xb)
            loss = cqr_loss(pred, yb, alpha)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item()
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"[Regressor] Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")
    qmodel.eval()

    # b) genero calibrazione e test attaccate con LO STESSO procedimento
    def _attack(X, y):
        if attack_name.upper() == "WORST":
            return generate_worst_case_examples(
                model=model_rb, X=X, y=y, get_true_probabilities=get_true_probabilities,
                attack_names=worst_case_pool, eps=eps, steps=steps, bs=bs, device=device
            )
        return generate_adversarial_examples(
            model=model_rb, X=X, y=y, attack_name=attack_name,
            eps=eps, steps=steps, bs=bs, device=device
        )

    X_calib_adv = _attack(X_calib, y_calib)
    X_test_adv = _attack(X_test, y_test)

    p_calib_clean = get_true_probabilities(model_rb, X_calib, y_calib).to(device).view(-1)
    p_calib_adv = get_true_probabilities(model_rb, X_calib_adv, y_calib).to(device).view(-1)
    p_test_adv = get_true_probabilities(model_rb, X_test_adv, y_test).to(device).view(-1)

    with torch.no_grad():
        pred_calib_clean = qmodel(X_calib.to(device))
        pred_calib_adv = qmodel(X_calib_adv.to(device))
        pred_test_adv = qmodel(X_test_adv.to(device))

    # c) VANILLA
    scores_clean = torch.maximum(pred_calib_clean[:, 0] - p_calib_clean,
                                  p_calib_clean - pred_calib_clean[:, 1])
    qhat_vanilla = torch.quantile(scores_clean, 1 - alpha)
    lo_v, hi_v = pred_test_adv[:, 0] - qhat_vanilla, pred_test_adv[:, 1] + qhat_vanilla
    coverage_v = ((p_test_adv >= lo_v) & (p_test_adv <= hi_v)).float().mean()
    size_v = (hi_v - lo_v).mean()

    # d) PERCP
    scores_adv = torch.maximum(pred_calib_adv[:, 0] - p_calib_adv,
                                p_calib_adv - pred_calib_adv[:, 1])
    qhat_percp = torch.quantile(scores_adv, 1 - alpha)
    lo_p, hi_p = pred_test_adv[:, 0] - qhat_percp, pred_test_adv[:, 1] + qhat_percp
    coverage_p = ((p_test_adv >= lo_p) & (p_test_adv <= hi_p)).float().mean()
    size_p = (hi_p - lo_p).mean()

    return {
        "attack": attack_name,
        "coverage_vanilla": coverage_v.item(), "size_vanilla": size_v.item(),
        "coverage_percp": coverage_p.item(), "size_percp": size_p.item(),
        "qhat_vanilla": qhat_vanilla.item(), "qhat_percp": qhat_percp.item(),
    }



In [11]:


# ----------------------------------------------------------------------
# Esempio d'uso end-to-end su un dataset a scelta
# ----------------------------------------------------------------------

# (basta che tu gli passi generate_adversarial_examples/generate_worst_case_examples
#  di QUESTO file al posto di quelli vecchi, oppure copi la funzione qui - la logica
#  interna di evaluate_attack_percp_v2 non cambia, chiama solo generate_adversarial_examples
#  con attack_name e eps)

DATASET = "CIFAR10"     # "CIFAR10", "CIFAR100", "IMAGENET"
eps = DATASET_DEFAULT_EPS_LINF[DATASET]

if DATASET == "CIFAR10":
   
    root = "/home/acarlevaro/Sources/albi/data"
    
elif DATASET == "CIFAR100":
    
    root = "/home/acarlevaro/Sources/albi/OLD/Extension/CIFAR100/data"
    
elif DATASET == "IMAGENET":
    
    root = "/home/acarlevaro/Sources/albi/OLD/Adversarial_CP_V3/InyImageNet/ILSVRC2012_Albi"
        
dataset = load_dataset(DATASET, root=root)
model_rb = load_robust_model(
    model_name="Ding2020MMA",       # vedi note in load_robust_model per dataset != cifar10
    dataset=DATASET,
    threat_model="Linf",
)

#IMAGENET: Xu2024MIMIR_Swin-L, Salman2020Dorn-18
#CIFAR100: Wang2023Better_WRN-70-16, Rice2020Overfitting
#CIFAR10: Bartoldson2024Adversarial_WRN-94-16,Ding2020MMA

In [12]:
regressor = RobustBenchProbabilityRegressor(model_rb).cuda()

idx = torch.randperm(len(dataset))
idx_learn, idx_test = idx[:1000], idx[1000:2000]
X_learn, y_learn, p_learn = get_probability_targets(regressor, dataset, idx_learn)
X_test, y_test, p_test = get_probability_targets(regressor, dataset, idx_test)
X_train, y_train, X_calib, y_calib = split_train_calib(X_learn, y_learn, frac_train=0.5, seed=0)

BATTERY = ["FGSM", "BIM", "PGD", "APGD", "FAB", "CW", "DEEPFOOL", "WORST"]#"AA"
results = []
for attack in BATTERY:
    res = evaluate_attack_percp_v2(
        attack_name=attack,
        model_rb=model_rb,
        QuantileRegressor=QuantileRegressor,
        ProbabilityRegressionDataset=ProbabilityRegressionDataset,
        cqr_loss=cqr_loss,
        get_true_probabilities=get_true_probabilities,
        X_train=X_train, y_train=y_train,
        X_calib=X_calib, y_calib=y_calib,
        X_test=X_test, y_test=y_test,
        eps=eps,
        alpha=0.1,
        steps=20,
        bs=16 if DATASET != "IMAGENET" else 4,   # immagini piu' grandi -> batch piu' piccolo
        worst_case_pool=["PGD", "APGD", "CW"], 
    )
    results.append(res)


Running FGSM (PERCP corretto)
[Regressor] Epoch 1/50 - Loss: 0.4486
[Regressor] Epoch 10/50 - Loss: 0.0772
[Regressor] Epoch 20/50 - Loss: 0.0628
[Regressor] Epoch 30/50 - Loss: 0.0480
[Regressor] Epoch 40/50 - Loss: 0.0363
[Regressor] Epoch 50/50 - Loss: 0.0296


FGSM: 100%|██████████| 63/63 [00:00<00:00, 71.87it/s]



Running BIM (PERCP corretto)
[Regressor] Epoch 1/50 - Loss: 0.6690
[Regressor] Epoch 10/50 - Loss: 0.0820
[Regressor] Epoch 20/50 - Loss: 0.0661
[Regressor] Epoch 30/50 - Loss: 0.0451
[Regressor] Epoch 40/50 - Loss: 0.0387
[Regressor] Epoch 50/50 - Loss: 0.0308


BIM: 100%|██████████| 63/63 [00:15<00:00,  4.15it/s]



Running PGD (PERCP corretto)
[Regressor] Epoch 1/50 - Loss: 0.2768
[Regressor] Epoch 10/50 - Loss: 0.0818
[Regressor] Epoch 20/50 - Loss: 0.0680
[Regressor] Epoch 30/50 - Loss: 0.0524
[Regressor] Epoch 40/50 - Loss: 0.0390
[Regressor] Epoch 50/50 - Loss: 0.0315


...


--- Worst-case ensemble: genero APGD ---


APGD: 100%|██████████| 63/63 [00:19<00:00,  3.18it/s]



--- Worst-case ensemble: genero CW ---


CW: 100%|██████████| 63/63 [00:21<00:00,  2.93it/s]



Distribuzione dell'attacco vincente per punto:
  PGD: 72.7%
  APGD: 27.2%
  CW: 0.1%


In [13]:

df = pd.DataFrame(results)

df["coverage_vanilla"] *= 100
df["coverage_percp"] *= 100

df 

,attack,coverage_vanilla,size_vanilla,coverage_percp,size_percp,qhat_vanilla,qhat_percp
0,FGSM,71.900004,0.650660,92.600006,1.511753,0.043256,0.473802
1,BIM,60.500002,0.578928,91.900003,1.743102,0.075232,0.657319
2,PGD,53.700000,0.586046,91.800004,1.850205,0.048860,0.680939
3,APGD,66.200006,0.750658,90.300006,1.860381,0.188058,0.742919
4,FAB,70.800000,0.611088,90.100002,1.136591,0.017332,0.280083
5,CW,78.200006,0.649806,89.300007,1.049344,0.063377,0.263146
6,DEEPFOOL,82.300001,0.588106,92.100006,0.860870,0.018966,0.155348
7,WORST,50.100005,0.499433,87.600005,1.871727,0.080605,0.766752


In [6]:

df = pd.DataFrame(results)

df["coverage_vanilla"] *= 100
df["coverage_percp"] *= 100

df 

,attack,coverage_vanilla,size_vanilla,coverage_percp,size_percp,qhat_vanilla,qhat_percp
0,FGSM,70.700002,0.643635,89.200002,1.143427,0.087570,0.337466
1,BIM,66.100001,0.686283,89.200002,1.247376,0.013407,0.293953
2,PGD,66.600001,0.704013,90.500003,1.323817,0.067692,0.377594
3,APGD,75.200003,0.763880,91.400003,1.426372,0.112277,0.443523
4,FAB,76.900005,0.701531,91.200006,1.094138,0.135932,0.332235
5,CW,81.400001,0.703936,91.600007,0.973466,0.059825,0.194590
6,DEEPFOOL,86.200005,0.681720,91.300005,0.798204,0.026929,0.085171
7,AA,71.600002,0.635693,88.900006,1.230486,0.130340,0.427737
8,WORST,62.200004,0.682924,88.900006,1.398401,0.075865,0.433603


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from secml.utils import fm
from secml import settings

import numpy as np

from robustbench.utils import load_model

import matplotlib.pyplot as plt

import torchattacks

from tqdm import tqdm

from nonconformist.cp import IcpRegressor
from nonconformist.nc import RegressorNc, QuantileRegErrFunc

import foolbox as fb

2026-07-20 11:15:24.606458: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-20 11:15:24.698653: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-20 11:15:26.764638: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


# DATASET

In [2]:
transform=T.ToTensor()

dataset=torchvision.datasets.CIFAR10(
    root="/home/acarlevaro/Sources/albi/data",
    train=True,
    download=True,
    transform=transform
)

# LOAD RB MODEL

In [3]:
output_dir = fm.join(settings.SECML_MODELS_DIR, 'robustbench')

model_rb = load_model(
    model_name="Wang2023Better_WRN-70-16", #Wang2023Better_WRN-70-16, Ding2020MMA
    norm="L2",
    model_dir=output_dir
)

model_rb.eval()

model_rb.cuda()

DMWideResNet(
  (init_conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (layer): Sequential(
    (0): _BlockGroup(
      (block): Sequential(
        (0): _Block(
          (batchnorm_0): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu_0): SiLU()
          (conv_0): Conv2d(16, 256, kernel_size=(3, 3), stride=(1, 1), bias=False)
          (batchnorm_1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu_1): SiLU()
          (conv_1): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (shortcut): Conv2d(16, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        )
        (1): _Block(
          (batchnorm_0): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu_0): SiLU()
          (conv_0): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), bias=False)
          (batchnorm

### Rregresion wrapper

In [4]:
class RobustBenchProbabilityRegressor(nn.Module):

    def __init__(self, classifier):
        super().__init__()
        self.classifier = classifier


    def forward(self, x, y):

        logits = self.classifier(x)

        probs = F.softmax(logits, dim=1)

        idx = torch.arange(
            len(y),
            device=x.device
        )

        p_true = probs[idx,y]

        return p_true

In [5]:
regressor = RobustBenchProbabilityRegressor(
    model_rb
).cuda()

### Get probability targets

In [6]:
def get_probability_targets(model,dataset,indices):

    X=[]
    y=[]
    p=[]


    with torch.no_grad():

        for i in indices:

            img,label=dataset[i]

            img=img.unsqueeze(0).cuda()
            label=torch.tensor(
                [label],
                device="cuda"
            )


            prob=model(
                img,
                label
            )


            X.append(img.cpu())
            y.append(label.cpu())
            p.append(prob.cpu())


    return (
        torch.cat(X),
        torch.cat(y),
        torch.cat(p)
    )

# ATTACKS

In [8]:
def get_true_probabilities(
    classifier,
    X,
    y,
    device="cuda"
):

    classifier.eval()

    probs_all=[]

    batch_size=64

    with torch.no_grad():

        for i in range(0,len(X),batch_size):

            x=X[i:i+batch_size].to(device)
            labels=y[i:i+batch_size].to(device)

            logits=classifier(x)

            probs=torch.softmax(
                logits,
                dim=1
            )

            p_true=probs[
                torch.arange(len(labels),device=device),
                labels
            ]

            probs_all.append(
                p_true.cpu()
            )

    return torch.cat(probs_all)

In [9]:
def split_train_calib(X_cal, y_cal, frac_train=0.5, seed=None):
    """Divide il pool 'X_cal' in due parti disgiunte:
    - X_train/y_train: usati per fittare il quantile regressor
    - X_calib/y_calib: usati SOLO per calcolare qhat (mai visti in training)
    """
    if seed is not None:
        torch.manual_seed(seed)
 
    n = len(X_cal)
    perm = torch.randperm(n)
    n_train = int(n * frac_train)
 
    idx_train = perm[:n_train]
    idx_calib = perm[n_train:]
 
    return (
        X_cal[idx_train], y_cal[idx_train],
        X_cal[idx_calib], y_cal[idx_calib],
    )
 

from torch.utils.data import Dataset, DataLoader


class ProbabilityRegressionDataset(Dataset):

    def __init__(self, X, y):
        self.X = X
        self.y = y.float()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return (
            self.X[idx],
            self.y[idx]
        )
    
import torchvision.models as models


class QuantileRegressor(nn.Module):

    def __init__(self):

        super().__init__()

        self.model = models.resnet18(
            weights=None
        )

        self.model.fc = nn.Linear(
            self.model.fc.in_features,
            2
        )


    def forward(self,x):

        out = self.model(x)

        return out
    
def pinball_loss(pred, target, quantile):

    error = target - pred

    return torch.mean(
        torch.maximum(
            quantile*error,
            (quantile-1)*error
        )
    )


def cqr_loss(pred, target, alpha):
    
    q_low=alpha/2
    q_high=1-alpha/2

    loss_low = pinball_loss(
        pred[:,0],
        target,
        q_low
    )

    loss_high = pinball_loss(
        pred[:,1],
        target,
        q_high
    )

    return loss_low + loss_high    
    
# ----------------------------------------------------------------------
# 1) PGD con steps configurabile (era hardcodato a 1 in generate_adversarial_examples)
# ----------------------------------------------------------------------
def pgd_attack(model, X, y, eps=0.5, norm="LINF", steps=100, device="cuda", batch_size=32):
    model.eval()
    norm = norm.upper()
 
    if norm == "L2":
        alpha = (eps / steps) * 2
        atk = torchattacks.PGDL2(model, eps=eps, alpha=alpha, steps=steps, random_start=True)
    elif norm == "LINF":
        alpha = (eps / steps) * 2
        atk = torchattacks.PGD(model, eps=eps, alpha=alpha, steps=steps, random_start=True)
    else:
        raise ValueError(f"Norma {norm} non supportata. Usa 'L2' o 'LINF'.")
 
    X_adv = []
    for i in tqdm(range(0, len(X), batch_size), desc=f"Running PGD ({norm})", leave=True):
        x_batch = X[i:i + batch_size].to(device)
        y_batch = y[i:i + batch_size].to(device)
        adv = atk(x_batch, y_batch)
        X_adv.append(adv.detach().cpu())
 
    return torch.cat(X_adv)
 
 
def generate_adversarial_examples(model, X, y, attack_name, eps, norm="LINF",
                                   steps=100, bs=128, device="cuda"):
    attack_name = attack_name.upper()
    if attack_name == "PGD":
        return pgd_attack(model=model, X=X, y=y, eps=eps, norm=norm,
                           steps=steps, device=device, batch_size=bs)
    raise NotImplementedError(f"{attack_name} not implemented yet")
 
 
# ----------------------------------------------------------------------
# 2) evaluate_attack corretto: UN SOLO regressore, split train/calib rispettato
# ----------------------------------------------------------------------
def evaluate_attack_percp(
    attack_name,
    model_rb,
    QuantileRegressor,          # classe del regressore (import dal tuo notebook)
    ProbabilityRegressionDataset,
    cqr_loss,
    get_true_probabilities,
    X_train, y_train,
    X_calib, y_calib,
    X_test, y_test,
    eps=0.5,
    alpha=0.1,
    pgd_steps=100,
    bs=16,
    norm="LINF",
    epochs=50,
    device="cuda",
):
    print(f"\nRunning {attack_name}")
    

    # --------------------------------------------------------------
    # a) Un solo regressore, allenato SOLO su X_train (mai in calibrazione)
    # --------------------------------------------------------------
    p_train = get_true_probabilities(model_rb, X_train, y_train).view(-1)
 
    train_loader = DataLoader(
        ProbabilityRegressionDataset(X_train, p_train.cpu()),
        batch_size=32, shuffle=True
    )
 
    qmodel = QuantileRegressor().to(device)
    optimizer = torch.optim.Adam(qmodel.parameters(), lr=1e-4)
 
    for epoch in range(epochs):
        qmodel.train()
        total_loss = 0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            pred = qmodel(Xb)
            loss = cqr_loss(pred, yb, alpha)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"[Regressor] Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")
 
    qmodel.eval()
 
    # --------------------------------------------------------------
    # b) Genero le versioni attaccate di CALIBRAZIONE e TEST con la
    #    STESSA procedura (stesso threat model per entrambe)
    # --------------------------------------------------------------
    X_calib_adv = generate_adversarial_examples(
        model=model_rb, X=X_calib, y=y_calib, attack_name=attack_name,
        eps=eps, norm=norm, steps=pgd_steps, bs=bs, device=device
    )
    X_test_adv = generate_adversarial_examples(
        model=model_rb, X=X_test, y=y_test, attack_name=attack_name,
        eps=eps, norm=norm, steps=pgd_steps, bs=bs, device=device
    )
 
    p_calib_clean = get_true_probabilities(model_rb, X_calib, y_calib).to(device).view(-1)
    p_calib_adv = get_true_probabilities(model_rb, X_calib_adv, y_calib).to(device).view(-1)
    p_test_adv = get_true_probabilities(model_rb, X_test_adv, y_test).to(device).view(-1)
 
    with torch.no_grad():
        pred_calib_clean = qmodel(X_calib.to(device))
        pred_calib_adv = qmodel(X_calib_adv.to(device))
        pred_test_adv = qmodel(X_test_adv.to(device))
 
    # --------------------------------------------------------------
    # c) VANILLA: qhat calcolato su calibrazione PULITA, stesso qmodel
    # --------------------------------------------------------------
    scores_clean = torch.maximum(
        pred_calib_clean[:, 0] - p_calib_clean,
        p_calib_clean - pred_calib_clean[:, 1]
    )
    qhat_vanilla = torch.quantile(scores_clean, 1 - alpha)
 
    lo_v = pred_test_adv[:, 0] - qhat_vanilla
    hi_v = pred_test_adv[:, 1] + qhat_vanilla
    coverage_v = ((p_test_adv >= lo_v) & (p_test_adv <= hi_v)).float().mean()
    size_v = (hi_v - lo_v).mean()
 
    # --------------------------------------------------------------
    # d) PERCP: STESSO qmodel, ma qhat ricalibrato sullo score calcolato
    #    su calibrazione ATTACCATA con lo stesso threat model del test
    # --------------------------------------------------------------
    scores_adv = torch.maximum(
        pred_calib_adv[:, 0] - p_calib_adv,
        p_calib_adv - pred_calib_adv[:, 1]
    )
    qhat_percp = torch.quantile(scores_adv, 1 - alpha)
 
    lo_p = pred_test_adv[:, 0] - qhat_percp
    hi_p = pred_test_adv[:, 1] + qhat_percp
    coverage_p = ((p_test_adv >= lo_p) & (p_test_adv <= hi_p)).float().mean()
    size_p = (hi_p - lo_p).mean()
 
    return {
        "attack": attack_name,
        "coverage_vanilla": coverage_v.item(),
        "size_vanilla": size_v.item(),
        "coverage_percp": coverage_p.item(),
        "size_percp": size_p.item(),
        "qhat_vanilla": qhat_vanilla.item(),
        "qhat_percp": qhat_percp.item(),
    }
 
 


In [16]:

idx=torch.randperm(len(dataset))

idx_learn=idx[:1000]
idx_test=idx[1000:2000]

X_learn, y_learn, p_learn = get_probability_targets(regressor,dataset,idx_learn)

X_test, y_test, p_test = get_probability_targets(regressor,dataset,idx_test)

X_train, y_train, X_calib, y_calib = split_train_calib(X_learn, y_learn, frac_train=0.5, seed=0)

In [13]:
ATTACKS = ["PGD"]
results = []
 
for attack in ATTACKS:
    res = evaluate_attack_percp(
        attack_name=attack,
        model_rb=model_rb,
        QuantileRegressor=QuantileRegressor,
        ProbabilityRegressionDataset=ProbabilityRegressionDataset,
        cqr_loss=cqr_loss,
        get_true_probabilities=get_true_probabilities,
        X_train=X_train, y_train=y_train,
        X_calib=X_calib, y_calib=y_calib,
        X_test=X_test, y_test=y_test,
        eps=0.5,       
        alpha=0.1,
        pgd_steps=20,     
        bs=16,
        norm="L2",
    )
    results.append(res)

import pandas as pd

df = pd.DataFrame(results)

df["coverage_vanilla"] *= 100
df["coverage_percp"] *= 100

df 


Running PGD
[Regressor] Epoch 1/50 - Loss: 0.4171
[Regressor] Epoch 10/50 - Loss: 0.0793
[Regressor] Epoch 20/50 - Loss: 0.0574
[Regressor] Epoch 30/50 - Loss: 0.0430
[Regressor] Epoch 40/50 - Loss: 0.0331
[Regressor] Epoch 50/50 - Loss: 0.0243


Running PGD (L2): 100%|██████████| 63/63 [00:14<00:00,  4.50it/s]


,attack,coverage_vanilla,size_vanilla,coverage_percp,size_percp,qhat_vanilla,qhat_percp
0,PGD,82.200003,0.296841,92.500007,0.557361,-0.043612,0.086648


In [10]:
def split_train_calib(X_cal, y_cal, frac_train=0.5, seed=None):
    """Divide il pool 'X_cal' in due parti disgiunte:
    - X_train/y_train: usati per fittare il quantile regressor
    - X_calib/y_calib: usati SOLO per calcolare qhat (mai visti in training)
    """
    if seed is not None:
        torch.manual_seed(seed)
 
    n = len(X_cal)
    perm = torch.randperm(n)
    n_train = int(n * frac_train)
 
    idx_train = perm[:n_train]
    idx_calib = perm[n_train:]
 
    return (
        X_cal[idx_train], y_cal[idx_train],
        X_cal[idx_calib], y_cal[idx_calib],
    )
 

from torch.utils.data import Dataset, DataLoader

class ProbabilityRegressionDataset(Dataset):

    def __init__(self, X, y):
        self.X = X
        self.y = y.float()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return (
            self.X[idx],
            self.y[idx]
        )
    
import torchvision.models as models


class QuantileRegressor(nn.Module):

    def __init__(self):

        super().__init__()

        self.model = models.resnet18(
            weights=None
        )

        self.model.fc = nn.Linear(
            self.model.fc.in_features,
            2
        )


    def forward(self,x):

        out = self.model(x)

        return out
    
def pinball_loss(pred, target, quantile):

    error = target - pred

    return torch.mean(
        torch.maximum(
            quantile*error,
            (quantile-1)*error
        )
    )


def cqr_loss(pred, target, alpha):
    
    q_low=alpha/2
    q_high=1-alpha/2

    loss_low = pinball_loss(
        pred[:,0],
        target,
        q_low
    )

    loss_high = pinball_loss(
        pred[:,1],
        target,
        q_high
    )

    return loss_low + loss_high    

def get_true_probabilities(
    classifier,
    X,
    y,
    device="cuda"
):

    classifier.eval()

    probs_all=[]

    batch_size=64

    with torch.no_grad():

        for i in range(0,len(X),batch_size):

            x=X[i:i+batch_size].to(device)
            labels=y[i:i+batch_size].to(device)

            logits=classifier(x)

            probs=torch.softmax(
                logits,
                dim=1
            )

            p_true=probs[
                torch.arange(len(labels),device=device),
                labels
            ]

            probs_all.append(
                p_true.cpu()
            )

    return torch.cat(probs_all)

In [11]:
idx=torch.randperm(len(dataset))

idx_learn=idx[:1000]
idx_test=idx[1000:2000]

X_learn, y_learn, p_learn = get_probability_targets(regressor,dataset,idx_learn)

X_test, y_test, p_test = get_probability_targets(regressor,dataset,idx_test)

X_train, y_train, X_calib, y_calib = split_train_calib(X_learn, y_learn, frac_train=0.5, seed=0)

NameError: name 'regressor' is not defined

In [8]:
import torch
import torchattacks
from tqdm import tqdm

try:
    import foolbox as fb
    _HAS_FOOLBOX = True
except ImportError:
    _HAS_FOOLBOX = False

try:
    from adv_lib.attacks import alma as _alma_fn
    _HAS_ADVLIB = True
except ImportError:
    _HAS_ADVLIB = False


# ----------------------------------------------------------------------
# Proiezione sulla palla Lp di raggio eps intorno a x_orig
# ----------------------------------------------------------------------
def project_lp_ball(x_adv, x_orig, eps, norm="LINF"):
    delta = x_adv - x_orig
    norm = norm.upper()
    if norm == "LINF":
        delta = delta.clamp(-eps, eps)
    elif norm == "L2":
        flat = delta.view(delta.size(0), -1)
        n = flat.norm(p=2, dim=1).clamp(min=1e-12)
        factor = (eps / n).clamp(max=1.0)
        delta = (flat * factor.unsqueeze(1)).view_as(delta)
    else:
        raise ValueError(f"Norma {norm} non supportata")
    return (x_orig + delta).clamp(0, 1)


def _batched(atk, x, y, device, bs, desc=None):
    out = []
    for i in tqdm(range(0, len(x), bs), desc=desc or atk.__class__.__name__, leave=True):
        xb, yb = x[i:i + bs].to(device), y[i:i + bs].to(device)
        out.append(atk(xb, yb).detach().cpu())
    return torch.cat(out)


# ----------------------------------------------------------------------
# Attacchi a budget fisso (rispettano eps ad ogni step)
# ----------------------------------------------------------------------
def _run_pgd(model, x, y, eps, norm, steps, device, bs):
    alpha = (eps / steps) * 2
    atk = (torchattacks.PGDL2(model, eps=eps, alpha=alpha, steps=steps, random_start=True)
           if norm.upper() == "L2"
           else torchattacks.PGD(model, eps=eps, alpha=alpha, steps=steps, random_start=True))
    return _batched(atk, x, y, device, bs, desc="PGD")


def _run_apgd(model, x, y, eps, norm, steps, device, bs):
    atk = torchattacks.APGD(model, norm=norm.capitalize(), eps=eps, steps=steps, loss="ce")
    return _batched(atk, x, y, device, bs, desc="APGD")


def _run_aa(model, x, y, eps, norm, steps, device, bs):
    # AutoAttack = APGD-CE + APGD-T + FAB-T + Square: e' gia' un ensemble,
    # 'steps' qui non e' un parametro diretto (usa i default della libreria).
    atk = torchattacks.AutoAttack(model, norm=norm.capitalize(), eps=eps, version="standard")
    return _batched(atk, x, y, device, bs, desc="AutoAttack")


# ----------------------------------------------------------------------
# Attacchi a norma minima (proiettati sulla palla dopo l'attacco)
# ----------------------------------------------------------------------
def _run_fab(model, x, y, eps, norm, steps, device, bs):
    atk = torchattacks.FAB(model, norm=norm.capitalize(), eps=eps, steps=steps,
                            n_restarts=1, n_classes=10)
    x_adv = _batched(atk, x, y, device, bs, desc="FAB")
    return project_lp_ball(x_adv, x, eps, norm)


def _run_cw(model, x, y, eps, norm, steps, device, bs):
    atk = torchattacks.CW(model, c=1, kappa=0, steps=steps, lr=0.01)
    x_adv = _batched(atk, x, y, device, bs, desc="CW")
    return project_lp_ball(x_adv, x, eps, norm)


def _run_ead(model, x, y, eps, norm, steps, device, bs):
    atk = torchattacks.EADL1(model, kappa=0, lr=0.01, max_iterations=steps)
    x_adv = _batched(atk, x, y, device, bs, desc="EAD")
    return project_lp_ball(x_adv, x, eps, norm)


def _run_ddn(model, x, y, eps, norm, steps, device, bs):
    if not _HAS_FOOLBOX:
        raise ImportError("DDN richiede foolbox: pip install foolbox")
    fmodel = fb.PyTorchModel(model, bounds=(0, 1))
    atk = fb.attacks.DDNAttack(steps=steps)
    out = []
    for i in tqdm(range(0, len(x), bs), desc="DDN", leave=True):
        xb, yb = x[i:i + bs].to(device), y[i:i + bs].to(device)
        _, clipped, _ = atk(fmodel, xb, yb, epsilons=eps)  # foolbox proietta gia' a eps
        out.append(clipped.detach().cpu())
    return torch.cat(out)


def _run_fmn(model, x, y, eps, norm, steps, device, bs):
    if not _HAS_FOOLBOX:
        raise ImportError("FMN richiede foolbox: pip install foolbox")
    fmodel = fb.PyTorchModel(model, bounds=(0, 1))
    atk = (fb.attacks.LInfFMNAttack(steps=steps) if norm.upper() == "LINF"
           else fb.attacks.L2FMNAttack(steps=steps))
    out = []
    for i in tqdm(range(0, len(x), bs), desc="FMN", leave=True):
        xb, yb = x[i:i + bs].to(device), y[i:i + bs].to(device)
        _, clipped, _ = atk(fmodel, xb, yb, epsilons=eps)
        out.append(clipped.detach().cpu())
    return torch.cat(out)


def _run_alma(model, x, y, eps, norm, steps, device, bs):
    if not _HAS_ADVLIB:
        raise ImportError(
            "ALMA richiede adv_lib: "
            "pip install git+https://github.com/jeromerony/adversarial-library"
        )
    norm_map = {"L2": 2, "LINF": float("inf")}
    model.to(device).eval()
    out = []
    for i in tqdm(range(0, len(x), bs), desc="ALMA", leave=True):
        xb, yb = x[i:i + bs].to(device), y[i:i + bs].to(device)
        # NOTA: la firma esatta puo' variare tra versioni di adv_lib.
        # Verifica con help(alma) se questo non dovesse funzionare cosi' com'e'.
        adv = _alma_fn(model=model, inputs=xb, labels=yb, norm=norm_map[norm.upper()])
        out.append(adv.detach().cpu())
    x_adv = torch.cat(out)
    return project_lp_ball(x_adv, x, eps, norm)


ATTACK_FUNCS = {
    "PGD": _run_pgd,
    "APGD": _run_apgd,
    "AA": _run_aa,
    "FAB": _run_fab,
    "CW": _run_cw,
    "EAD": _run_ead,
    "DDN": _run_ddn,
    "FMN": _run_fmn,
    "ALMA": _run_alma,
}


def generate_adversarial_examples(model, X, y, attack_name, eps, norm="LINF",
                                   steps=100, bs=16, device="cuda"):
    """Dispatcher unico per tutti gli attacchi della batteria."""
    attack_name = attack_name.upper()
    if attack_name not in ATTACK_FUNCS:
        raise NotImplementedError(
            f"{attack_name} non riconosciuto. Disponibili: {list(ATTACK_FUNCS)} (+ 'WORST')"
        )
    return ATTACK_FUNCS[attack_name](model, X, y, eps, norm, steps, device, bs)


# ----------------------------------------------------------------------
# "Worst": per ogni punto tiene la versione che minimizza p_true
# ----------------------------------------------------------------------
def generate_worst_case_examples(model, X, y, get_true_probabilities,
                                  attack_names, eps, norm="LINF",
                                  steps=100, bs=16, device="cuda"):
    """Genera X_adv con ciascun attacco in attack_names, poi per ogni punto
    tiene la versione con la p_true(x) piu' bassa (= degradazione peggiore).
    Usare lo STESSO attack_names/eps/norm/steps per calibrazione e test.
    """
    candidates, p_candidates = [], []

    for name in attack_names:
        print(f"\n--- Worst-case ensemble: genero {name} ---")
        X_adv = generate_adversarial_examples(
            model=model, X=X, y=y, attack_name=name,
            eps=eps, norm=norm, steps=steps, bs=bs, device=device
        )
        p_adv = get_true_probabilities(model, X_adv, y).view(-1).cpu()
        candidates.append(X_adv)
        p_candidates.append(p_adv)

    P = torch.stack(p_candidates, dim=0)            # (n_attacks, N)
    worst_idx = P.argmin(dim=0)                      # (N,) attacco peggiore per punto

    X_stack = torch.stack(candidates, dim=0)          # (n_attacks, N, C, H, W)
    N = X.shape[0]
    X_worst = X_stack[worst_idx, torch.arange(N)]     # (N, C, H, W)

    print("\nDistribuzione dell'attacco vincente per punto:")
    for i, name in enumerate(attack_names):
        pct = (worst_idx == i).float().mean().item() * 100
        print(f"  {name}: {pct:.1f}%")

    return X_worst


# ----------------------------------------------------------------------
# evaluate_attack_percp esteso: gestisce anche attack_name="WORST"
# (riusa lo split train/calib e il singolo qmodel del fix precedente)
# ----------------------------------------------------------------------
def evaluate_attack_percp_v2(
    attack_name,                # una tra ATTACK_FUNCS oppure "WORST"
    model_rb,
    QuantileRegressor,
    ProbabilityRegressionDataset,
    cqr_loss,
    get_true_probabilities,
    X_train, y_train,
    X_calib, y_calib,
    X_test, y_test,
    eps=8 / 255,
    alpha=0.1,
    steps=100,
    bs=16,
    norm="LINF",
    epochs=50,
    device="cuda",
    worst_case_pool=("PGD", "APGD", "FAB", "CW"),  # sottoinsieme usato da "WORST"
):
    print(f"\nRunning {attack_name} (PERCP corretto)")

    # a) UN SOLO regressore, allenato SOLO su X_train
    p_train = get_true_probabilities(model_rb, X_train, y_train).view(-1)
    train_loader = torch.utils.data.DataLoader(
        ProbabilityRegressionDataset(X_train, p_train.cpu()), batch_size=32, shuffle=True
    )
    qmodel = QuantileRegressor().to(device)
    optimizer = torch.optim.Adam(qmodel.parameters(), lr=1e-4)
    for epoch in range(epochs):
        qmodel.train()
        total_loss = 0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            pred = qmodel(Xb)
            loss = cqr_loss(pred, yb, alpha)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item()
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"[Regressor] Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")
    qmodel.eval()

    # b) genero calibrazione e test attaccate con LO STESSO procedimento
    def _attack(X, y):
        if attack_name.upper() == "WORST":
            return generate_worst_case_examples(
                model=model_rb, X=X, y=y, get_true_probabilities=get_true_probabilities,
                attack_names=worst_case_pool, eps=eps, norm=norm, steps=steps, bs=bs, device=device
            )
        return generate_adversarial_examples(
            model=model_rb, X=X, y=y, attack_name=attack_name,
            eps=eps, norm=norm, steps=steps, bs=bs, device=device
        )

    X_calib_adv = _attack(X_calib, y_calib)
    X_test_adv = _attack(X_test, y_test)

    p_calib_clean = get_true_probabilities(model_rb, X_calib, y_calib).to(device).view(-1)
    p_calib_adv = get_true_probabilities(model_rb, X_calib_adv, y_calib).to(device).view(-1)
    p_test_adv = get_true_probabilities(model_rb, X_test_adv, y_test).to(device).view(-1)

    with torch.no_grad():
        pred_calib_clean = qmodel(X_calib.to(device))
        pred_calib_adv = qmodel(X_calib_adv.to(device))
        pred_test_adv = qmodel(X_test_adv.to(device))

    # c) VANILLA
    scores_clean = torch.maximum(pred_calib_clean[:, 0] - p_calib_clean,
                                  p_calib_clean - pred_calib_clean[:, 1])
    qhat_vanilla = torch.quantile(scores_clean, 1 - alpha)
    lo_v, hi_v = pred_test_adv[:, 0] - qhat_vanilla, pred_test_adv[:, 1] + qhat_vanilla
    coverage_v = ((p_test_adv >= lo_v) & (p_test_adv <= hi_v)).float().mean()
    size_v = (hi_v - lo_v).mean()

    # d) PERCP
    scores_adv = torch.maximum(pred_calib_adv[:, 0] - p_calib_adv,
                                p_calib_adv - pred_calib_adv[:, 1])
    qhat_percp = torch.quantile(scores_adv, 1 - alpha)
    lo_p, hi_p = pred_test_adv[:, 0] - qhat_percp, pred_test_adv[:, 1] + qhat_percp
    coverage_p = ((p_test_adv >= lo_p) & (p_test_adv <= hi_p)).float().mean()
    size_p = (hi_p - lo_p).mean()

    return {
        "attack": attack_name,
        "coverage_vanilla": coverage_v.item(), "size_vanilla": size_v.item(),
        "coverage_percp": coverage_p.item(), "size_percp": size_p.item(),
        "qhat_vanilla": qhat_vanilla.item(), "qhat_percp": qhat_percp.item(),
    }


# ----------------------------------------------------------------------
# Esempio d'uso
# ----------------------------------------------------------------------

BATTERY = ["PGD", "APGD", "FAB", "CW", "EAD", "DDN", "FMN", "AA", "WORST"]
# ALMA a parte se hai installato adv_lib

results = []
for attack in BATTERY:
    res = evaluate_attack_percp_v2(
        attack_name=attack,
        model_rb=model_rb,
        QuantileRegressor=QuantileRegressor,
        ProbabilityRegressionDataset=ProbabilityRegressionDataset,
        cqr_loss=cqr_loss,
        get_true_probabilities=get_true_probabilities,
        X_train=X_train, y_train=y_train,
        X_calib=X_calib, y_calib=y_calib,
        X_test=X_test, y_test=y_test,
        eps=0.5,
        alpha=0.1,
        steps=20,          # per la prima run tienilo basso, e' costoso con 9 attacchi
        bs=16,
        norm="L2",
        worst_case_pool=["PGD", "APGD", "FAB", "CW"],  # sottoinsieme, per contenere i tempi
    )
    results.append(res)
    df = pd.DataFrame(results)

    df["coverage_vanilla"] *= 100
    df["coverage_percp"] *= 100

    df


NameError: name 'X_train' is not defined

In [17]:

df = pd.DataFrame(results)

df["coverage_vanilla"] *= 100
df["coverage_percp"] *= 100

df 

,attack,coverage_vanilla,size_vanilla,coverage_percp,size_percp,qhat_vanilla,qhat_percp
0,PGD,84.000003,0.388217,88.200003,0.427246,0.023938,0.043452
1,APGD,90.700006,0.628934,90.700006,0.628934,-0.064197,-0.064197
2,FAB,91.500002,0.355069,91.500002,0.355069,0.013322,0.013322
3,CW,90.100002,0.404221,90.100002,0.406589,0.014067,0.015251
4,EAD,89.700001,0.455874,91.400003,0.469172,-0.029870,-0.023222
5,DDN,86.500007,0.381689,89.900005,0.425032,0.033364,0.055036
6,FMN,84.800005,0.352543,89.000005,0.394571,-0.000294,0.020721
